In [30]:
import yfinance as yf   # 야후 금융데이터 API
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN, # RNN 은닉층
    Dense, Input 
)
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [5]:
# 1. 삼성전자 최근 3년 주가 데이터 수집
df = yf.download(
    "005930.KS",
    period="3y",
    interval="1d",  # 종가
    auto_adjust=True,
    progress=False
)

df

Price,Close,High,Low,Open,Volume
Ticker,005930.KS,005930.KS,005930.KS,005930.KS,005930.KS
Date,,,,,
2023-10-23,65024.507812,65689.963302,64834.377673,65309.703022,11625959
2023-10-24,65119.574219,65404.769434,64359.053644,65309.704362,12791710
2023-10-25,64644.238281,65404.758732,64549.173225,65404.758732,10610703
2023-10-26,63408.406250,64549.187172,63408.406250,63693.601481,15517624
2023-10-27,63978.796875,63978.796875,63408.406413,63788.666721,11334726
...,...,...,...,...,...
2026-10-16,215500.000000,222500.000000,210500.000000,215000.000000,18908560
2026-10-19,218000.000000,219500.000000,206500.000000,207000.000000,17617415


In [ ]:
s = df['Close'] # (729,)    numpy배열로 변경하고 RNN입력값에 맞게 2차원 reshape
display(s)

#(759,1)    numpy배열 변경만
df = df[['Close']]
display(df)

Ticker,005930.KS
Date,
2023-10-23,65024.507812
2023-10-24,65119.574219
2023-10-25,64644.238281
2023-10-26,63408.406250
2023-10-27,63978.796875
...,...
2026-10-16,215500.000000
2026-10-19,218000.000000
2026-10-20,211500.000000


Price,Close
Ticker,005930.KS
Date,
2023-10-23,65024.507812
2023-10-24,65119.574219
2023-10-25,64644.238281
2023-10-26,63408.406250
2023-10-27,63978.796875
...,...
2026-10-16,215500.000000
2026-10-19,218000.000000


In [9]:
# 날짜 인덱스가 필요없다 0 ~ 728 였으면 한다
df = df.reset_index(drop=True) # drop=True 원래 인덱스를 새로운 열로 사용하지 않고 버림
df

Price,Close
Ticker,005930.KS
0,65024.507812
1,65119.574219
2,64644.238281
3,63408.406250
4,63978.796875
...,...
724,215500.000000
725,218000.000000
726,211500.000000


In [10]:
# DataFrame일 때 전처리하자 (ndarry 보다는 DataFrame 다루기에 편하고 익숙하기 때문)
# df.isna() True/False 729개 반환
df.isna().sum # True 1 / False 0으로 집계 -> 0이 아니면, 그 수가 결측치의 개수

<bound method DataFrame.sum of Price      Close
Ticker 005930.KS
0          False
1          False
2          False
3          False
4          False
..           ...
724        False
725        False
726        False
727        False
728        False

[729 rows x 1 columns]>

In [ ]:
# Series -> RNN 입력층 사용X

data = df['Close'].to_numpy()
print(data.shape)

(729, 1)


In [17]:
print(data[:5])
print(data[-5:])

[[65024.5078125 ]
 [65119.57421875]
 [64644.23828125]
 [63408.40625   ]
 [63978.796875  ]]
[[215500.]
 [218000.]
 [211500.]
 [223500.]
 [220250.]]


In [ ]:
# MinMaxScaler를 이용하여 0~1사이값 정규화. 신경망은 수치값의 차이가 작으면 작을수록 학습결과가 좋을 확률이 높다
scaler = MinMaxScaler()
data = scaler.fit_transform(data)

print(data[:5])
print(data[-5:])

[[0.05310928]
 [0.05341229]
 [0.05189722]
 [0.04795817]
 [0.04977621]]
[[0.5327299 ]
 [0.54069832]
 [0.51998043]
 [0.55822884]
 [0.5478699 ]]


In [24]:
# 스케일링(정규화)된 데이터를 가지고 모델 생성 -> 학습 -> 평가 -> 예측
# RNN : data의 변수 하나를 X, y 로 분리 -> X 입력값(2차원) -> 20일 주가로 21 주가를 예측 (20, 1)

w_size = 20

# 리스트 API를 이용해서 데이터를 생성 -> 배열로 변환
X = [] # [[1..... 20개], [], []]    2차원
y = [] # [1, 2, 5]  값


for i in range(len(data)- w_size): # 마지막 20일에 대해서는 y값이 존재하지 않기 때문
    X.append(data[i:i + w_size])
    y.append(data[i + w_size, 0]) # 열인덱스 지정해서 값이 저장되도록

print(X[0]) # 20일치 -> 20개의 변수
print(y[0]) # 21일 하나의 값

[[0.05310928]
 [0.05341229]
 [0.05189722]
 [0.04795817]
 [0.04977621]
 [0.04977621]
 [0.04856417]
 [0.05371528]
 [0.05704834]
 [0.05674532]
 [0.06068446]
 [0.06068446]
 [0.05765439]
 [0.05886641]
 [0.05947243]
 [0.05916941]
 [0.06038146]
 [0.06462352]
 [0.06644161]
 [0.06553254]]
0.06613858801333852


In [27]:
# X, y를 RNN이 사용하는 텐서 or 배열로 변경
X = np.array(X) 
y = np.array(y)

print(X.shape)
print(y.shape)

(709, 20, 1)
(709,)


In [28]:
# 709 Train / Validation -> train_test_split
from sklearn.model_selection import train_test_split

# train_test_split을 이용한 데이터 분할 (시간 순서 유지)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    shuffle=False  # 필수: 무작위 셔플 방지
)
print(X_train.shape)
print(X_test.shape)

(567, 20, 1)
(142, 20, 1)


In [31]:
# 모델 RNN
model = Sequential([
    Input(shape=(20, 1)),
    SimpleRNN(50),
    Dense(1)
])


In [32]:
# 손실 함수와 최적화 알고리즘 설정
model.compile(
    optimizer="adam",
    loss="mse"
)

In [33]:
model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    shuffle=False
)

Epoch 1/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.9040e-04   
Epoch 2/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0011     
Epoch 3/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0014     
Epoch 4/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0011     
Epoch 5/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0013     
Epoch 6/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0011     
Epoch 7/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 9.0039e-04 
Epoch 8/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.2279e-04 
Epoch 9/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.2817e-04 
Epoch 10/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.2665e-04 
Epoch 11/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.3347e-04 
Epoch 12/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.2899e-04 
Epoch 13/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1856e-04 
Epoch 14/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.2067e-04 
Epoch 15/50
1

In [36]:
pred = model.predict(X_test)
r2_score(y_test, pred)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


0.8775874698426672

In [40]:
last_20_days = data[-20:]
print(last_20_days.shape)   # 3차원 (20, 1) -> (1, 20, 1)

last_20_days = last_20_days.reshape(1, 20, 1)
print(last_20_days.shape)

(20, 1)
(1, 20, 1)


In [42]:
next_price = model.predict(last_20_days)
print(next_price)

print(scaler.inverse_transform(next_price))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
[[0.56226015]]
[[224764.78]]
